# Sales Forecasting Using Time Series Analysis

**Goal:** analyze historical daily revenue, compare forecasting models, and predict the next 30 days.

The included dataset is synthetic and is intended for practice, not as real company data. Run this notebook from the project root folder.

## 1. Import libraries and load the data

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "forecasting.py").exists():
    # Also support opening the notebook while the working directory is notebooks/
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from forecasting import load_and_clean_data, make_daily_series, train_and_compare, save_eda_charts

data_path = PROJECT_ROOT / "data" / "retail_sales.csv"
df_raw = pd.read_csv(data_path)
print("Rows and columns:", df_raw.shape)
display(df_raw.head())
display(df_raw.info())

## 2. Data quality checks and cleaning

In [ ]:
print("Missing values before cleaning:")
display(df_raw.isna().sum().to_frame("missing_count"))
print("Duplicate rows before cleaning:", df_raw.duplicated().sum())

df = load_and_clean_data(data_path)
print("Shape after cleaning:", df.shape)
print("Duplicate rows after cleaning:", df.duplicated().sum())
display(df.describe(include="all").T)

## 3. Create a daily revenue time series

In [ ]:
daily = make_daily_series(df)
display(daily.head())
print("Date range:", daily.index.min().date(), "to", daily.index.max().date())
print("Daily observations:", len(daily))

## 4. Exploratory data analysis (EDA)

In [ ]:
plt.figure(figsize=(13, 5))
daily.plot()
plt.title("Historical Daily Revenue")
plt.xlabel("Date"); plt.ylabel("Revenue"); plt.tight_layout()
plt.show()

monthly = daily.resample("MS").sum()
plt.figure(figsize=(13, 5))
monthly.plot(kind="bar")
plt.title("Monthly Revenue")
plt.xlabel("Month"); plt.ylabel("Revenue")
plt.xticks(rotation=60); plt.tight_layout()
plt.show()

In [ ]:
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekday_avg = df.groupby("Day_Of_Week")["Revenue"].mean().reindex(weekday_order)
weekday_avg.plot(kind="bar", figsize=(10, 4), title="Average Revenue by Weekday")
plt.ylabel("Average revenue"); plt.xticks(rotation=35); plt.tight_layout(); plt.show()

promo_avg = df.groupby("Promotion")["Revenue"].mean()
promo_avg.index = ["No Promotion" if i == 0 else "Promotion" for i in promo_avg.index]
promo_avg.plot(kind="bar", figsize=(7, 4), title="Average Revenue: Promotion Comparison")
plt.ylabel("Average revenue"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 5. Chronological train-test split and model comparison

The final 90 days are reserved for testing. This reflects forecasting better than randomly shuffling dates. We compare a naive baseline, a weekly seasonal-naive baseline, ARIMA, and Holt-Winters.

In [ ]:
train, test, metrics, predictions = train_and_compare(daily, test_days=90)
display(metrics)
best_model = metrics.iloc[0]["Model"]
print("Best model by RMSE:", best_model)

## 6. Actual vs predicted sales

In [ ]:
best_prediction = predictions[best_model]
plt.figure(figsize=(13, 5))
train.tail(180).plot(label="Training history")
test.plot(label="Actual", linewidth=2)
best_prediction.plot(label=f"Predicted — {best_model}", linestyle="--")
plt.title("Actual vs Predicted Revenue (Test Period)")
plt.xlabel("Date"); plt.ylabel("Revenue"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Forecast the next 30 days and save outputs

In [ ]:
# Run the full reusable pipeline. It saves forecast CSVs, charts, metrics, and the report.
from forecasting import run_pipeline
metrics_saved, future = run_pipeline()
display(future.head(10))
print("Saved outputs to:", PROJECT_ROOT / "outputs")

## 8. Business interpretation

Use the generated `reports/business_insights_report.md` and `outputs/model_metrics.csv` to discuss findings. Explain that:
- MAE and RMSE are measured in revenue units; lower is better.
- MAPE excludes rows where actual sales are zero.
- Promotional comparisons show association, not causation.
- Forecasts are estimates and should be checked against actual sales.
- The dataset is synthetic and must not be presented as real business data.